# Moving from Prototype to Production

Transitioning a generative AI prototype to a production-ready deployment involves careful consideration of model size, computational complexity, and robust evaluation. *Generative AI Foundations in Python* highlights the process of setting up a robust Python environment using Docker, CI/CD pipelines, and proper monitoring. 

Choosing the right pretrained generative model requires evaluating trade-offs between precision, speed, and resource constraints. It is also essential to have a strategy for evaluating model outputs. For example, using models like CLIP to evaluate the semantic alignment between text and images provides a quantitative measure of fidelity. Beyond metrics, production deployment must emphasize responsible AI practices, addressing transparency, explainability, and the mitigation of inherent biases.



# From Models to Systems: The Applied AI Gap {#sec-intro}

In prior modules we learned to build individual AI components: embeddings,
transformers, RAG pipelines, and agentic loops. But most real business problems
are not solved by isolated components — they are solved by **systems**: data flows
that ingest inputs, apply multiple AI models, and surface actionable decisions through
an interface that humans can actually use.

This module closes the loop. We study how the models we have built become
**production-grade pipelines** embedded in business dashboards and decision-support tools.

The questions we address:

1. How do you architect an end-to-end pipeline that connects raw data to ranked, explainable outputs?
2. How do you integrate semantic search, knowledge graphs, and LLMs without losing auditability?
3. How do you design for **explainability** — not as an afterthought, but as a first-class architectural constraint?

We anchor these questions in a concrete case study: **JobMatchAI**, a production
job-matching platform built at Boston University that integrates transformer embeddings,
knowledge graphs, hybrid retrieval, and LLM-based explanations into a single auditable
pipeline [@padalkar2025jobmatchai].


# The Challenge: From Keyword Search to Intelligent Matching {#sec-challenge}

## Why Keyword Search Fails

Most production search systems in HR, finance, legal, and operations still depend on
**lexical matching** (BM25, TF-IDF keyword overlap). These systems:

- Fail on **skill synonyms**: *Kubernetes* and *container orchestration* mean the same
  thing to a practitioner but do not share a single token.
- Miss **nonlinear career trajectories**: a candidate who spent 3 years as a data
  analyst and 2 years as an ML engineer has a combined profile that a keyword filter
  cannot represent.
- Produce **opaque scores**: a candidate is ranked #1 or #47 with no explanation,
  making the ranking impossible to audit.

In hiring, these failures have material and legal consequences. Missed candidates waste
recruiter time. Opaque rankings create regulatory risk — NYC Local Law 144 and related
AI-in-hiring regulations increasingly require that algorithmic scores be auditable.

## The Three Capability Gaps

| Gap | Description | Solution |
|-----|-------------|---------|
| **Semantic gap** | Lexical mismatch between job and candidate vocabulary | Dense embeddings + knowledge graph expansion |
| **Transparency gap** | Scores cannot be explained or audited | White-box utility function with named factors |
| **Integration gap** | Semantic, lexical, and graph signals not combined | Hybrid retrieval + reciprocal rank fusion |

These three gaps define the architecture of JobMatchAI — and more broadly, the design
challenges for any **AI-powered business dashboard**.


# Architecture of an AI Business Pipeline {#sec-architecture}

## The Five Layers

A production AI pipeline for business decision support typically comprises five layers:

```
Raw Data → Ingestion & Processing → Index Layer → Retrieval & Ranking → Explanation & UI
```

| Layer | Responsibility | Components |
|-------|---------------|------------|
| **Ingestion** | Fetch, normalize, embed documents | Crawlers, NLP parsers, Sentence Transformers |
| **Index** | Store structured + unstructured representations | Elasticsearch (BM25), vector store (ANN), knowledge graph (Neo4j) |
| **Retrieval** | Find candidate results from multiple signals | Parallel BM25 + semantic ANN + graph traversal |
| **Ranking** | Score and sort by business-relevant factors | Multi-factor utility function |
| **Explanation** | Surface reasoning to users | LLM narration grounded in pre-computed scores |

The key design principle: **push neural computation to ingestion time, keep online
serving lightweight**. Embeddings are computed once and stored; online query latency
is dominated by vector lookup and graph traversal (< 100 ms), not model inference.


## The JobMatchAI Architecture

JobMatchAI implements this five-layer stack as a set of stateless microservices behind
an API gateway (Figure 1):

**Ingestion pipeline**: Cron-scheduled crawlers fetch job postings from multiple sources.
An AI normalization layer standardizes formats. `all-MiniLM-L6-v2` (384-dimensional
Sentence Transformer) embeds each document. A spaCy-based extractor identifies skills,
locations, companies, and seniority level.

**Dual indexing**: Every document gets indexed twice — into Elasticsearch for BM25 lexical
search and kNN vector retrieval, and into Neo4j as a structured graph.

**Knowledge graph schema**: Five node types connected by typed edges:

```
(Candidate) -[:HAS_SKILL]-> (Skill) -[:RELATED_TO]-> (Skill) <-[:REQUIRES_SKILL]- (Job)
(Candidate) -[:LOCATED_IN]-> (Location) <-[:LOCATED_IN]- (Job)
(Candidate) -[:WORKED_AT]-> (Company) <-[:POSTED_BY]- (Job)
```

The `RELATED_TO` edge is the key innovation: it encodes skill associations (e.g.,
*Kubernetes ↔ Docker*, *scikit-learn ↔ machine learning*), enabling **multi-hop skill
discovery** — candidates who know Kubernetes match jobs requiring "container
orchestration" via a two-hop graph path.


# Hybrid Search: Combining Three Retrieval Signals {#sec-retrieval}

## Why One Signal Is Not Enough

No single retrieval approach dominates:

| Approach | Strength | Weakness |
|----------|---------|---------|
| **BM25 (lexical)** | Exact keyword matching, fast | Misses synonyms, semantic drift |
| **Dense ANN (semantic)** | Handles paraphrase and synonymy | May miss exact skill names |
| **Knowledge graph** | Semantic skill expansion, multi-hop | Limited to graph schema |

The solution is **hybrid retrieval**: run all three in parallel and fuse results.

## The Five-Stage Pipeline

**Stage 1 — Query Enrichment**: The raw query $q$ is expanded to
$\hat{q} = \langle E, S, S^+, \mathbf{e}_q, K \rangle$ where:
- $E$ = named entities extracted by spaCy
- $S$ = extracted skills
- $S^+$ = graph-expanded skills (depth-2 traversal of `RELATED_TO` edges)
- $\mathbf{e}_q$ = dense query embedding
- $K$ = keywords for BM25

**Stage 2 — Parallel Multi-Source Retrieval**: Three retrievers run concurrently:


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def parallel_retrieve(query, k_lex=150, k_sem=150, k_graph=75):
    with ThreadPoolExecutor(max_workers=3) as pool:
        f_lex   = pool.submit(elasticsearch_bm25,  query["keywords"],   k_lex)
        f_sem   = pool.submit(elasticsearch_knn,   query["embedding"],  k_sem)
        f_graph = pool.submit(neo4j_skill_traverse, query["skills_expanded"], k_graph)
    return {
        "lexical":   f_lex.result(),
        "semantic":  f_sem.result(),
        "graph":     f_graph.result(),
    }


**Stage 3 — Query-Adaptive Reciprocal Rank Fusion**:

$$
\begin{align}
\mathrm{RRF}(d) = \sum_{r \in R} \frac{w_r}{k + \mathrm{rank}_r(d)}, \qquad k = 60
\end{align}
$$

Weights $w_r$ adapt to query length: short queries (≤ 2 tokens) prioritize the knowledge
graph ($w_\text{KG} = 0.7$); longer free-text queries shift weight toward text
($w_\text{text} = 0.6$).

**Stage 4 — Hard-Constraint Filtering**: Post-retrieval filters remove candidates
who fail non-negotiable requirements (visa sponsorship, degree, required certifications)
before spending reranking budget.

**Stage 5 — Multi-Factor Reranking**: Final scoring by the utility function (see next
section).


# Explainable Reranking: The Utility Function {#sec-reranking}

## From Scores to Rankings

The final ranking is produced by a **white-box utility function**:

$$
\begin{align}
U(c, j) = \sum_{f \in \mathcal{F}} w_f \cdot \phi_f(c, j)
\end{align}
$$

where $\mathcal{F} = \{\text{Skill}, \text{Experience}, \text{Location}, \text{Salary}, \text{Semantic}, \text{Company}\}$
and each $\phi_f \in [0,1]$ is a normalized factor score.

| Factor $f$ | Default weight $w_f$ | Feature $\phi_f$ |
|---|:---:|---|
| Skill match | 0.35 | Jaccard overlap + knowledge-graph relatedness bonus |
| Experience | 0.25 | Level-distance penalty (junior/mid/senior) |
| Location | 0.15 | Tiered: exact / state / remote |
| Salary | 0.10 | Expectation-to-midpoint ratio |
| Semantic similarity | 0.10 | $\cos(\mathbf{e}_c, \mathbf{e}_j)$ |
| Company fit | 0.05 | Industry & size preference alignment |

**Weights sum to 1.0. Users can adjust weights interactively via frontend sliders;
the utility function recomputes in real time.**


In [ ]:
import numpy as np

def utility_score(candidate: dict, job: dict,
                  weights: dict = None) -> tuple[float, dict]:
    """
    Compute U(c, j) = sum_f w_f * phi_f(c, j).
    Returns (total_score, factor_breakdown).
    """
    if weights is None:
        weights = {"skill": 0.35, "experience": 0.25, "location": 0.15,
                   "salary": 0.10, "semantic": 0.10, "company": 0.05}

    factors = {
        "skill":      skill_score(candidate["skills"], job["required_skills"]),
        "experience": experience_score(candidate["level"], job["level"]),
        "location":   location_score(candidate["location"], job["location"]),
        "salary":     salary_score(candidate["salary_expect"], job["salary_range"]),
        "semantic":   cosine_sim(candidate["embedding"], job["embedding"]),
        "company":    company_score(candidate["prefs"], job["company_meta"]),
    }

    total = sum(weights[f] * factors[f] for f in factors)
    return round(total, 4), factors


def skill_score(candidate_skills: set, required_skills: set,
                kg_bonus: float = 0.1) -> float:
    """Jaccard similarity + knowledge-graph relatedness bonus."""
    if not required_skills:
        return 0.0
    intersection = candidate_skills & required_skills
    union = candidate_skills | required_skills
    jaccard = len(intersection) / len(union)
    # kg_bonus: fraction of required skills reachable via RELATED_TO edges
    return min(1.0, jaccard + kg_bonus)


## The Separation Principle: Scoring vs. Explanation

The most important architectural decision in JobMatchAI is the **strict separation of
scoring from narration**:

```
Scoring layer:  deterministic, auditable, no LLM
    ↓  (six pre-computed scores + KG paths)
Explanation layer:  LLM receives scores, not raw documents
    ↓
Natural-language narrative grounded in factor values
```

Because the LLM receives *pre-computed scores and knowledge-graph evidence* rather than
raw documents, it can **explain a ranking but cannot hallucinate a high one**.
The utility score is immutable; the LLM only narrates it.


In [ ]:
def generate_explanation(candidate: dict, job: dict,
                         score: float, factors: dict,
                         kg_paths: list[str]) -> str:
    """
    LLM explanation prompt — grounded in pre-computed scores only.
    The LLM cannot see raw resume or job description text.
    """
    factor_text = "\n".join(
        f"  - {k.capitalize()}: {v:.2f}" for k, v in factors.items()
    )
    kg_text = "\n".join(f"  • {p}" for p in kg_paths[:3])

    prompt = f"""
You are an HR assistant. Explain this job match in 2–3 sentences.
Do NOT change the scores. Use only the information below.

Match score: {score:.0%}

Factor breakdown:
{factor_text}

Knowledge graph evidence:
{kg_text}

Write a concise, factual explanation for the candidate.
"""
    return prompt   # in production, send to OpenAI / Anthropic API


Example output:
> *"This role is a strong match (87%) primarily due to verified expertise in Python and
> React. The location preference (Remote) conflicts slightly with the on-site requirement,
> reducing the location score."*


# Evaluation: Measuring System Quality {#sec-evaluation}

## The JobSearch-XS Benchmark

Evaluating search systems requires domain-specific benchmarks. The authors release
**JobSearch-XS**: 1,283 NYC civil-service job roles, 30 queries, 29K silver relevance
labels, and **skill-disjoint train/dev/test splits** to test zero-shot skill
generalization.

The disjoint splits are critical: if a skill appears in training, it should not appear
in test. This forces the system to generalize through graph traversal and semantic
embeddings — not memorization.

## Retrieval Metrics

$$
\begin{align}
\text{NDCG@}k = \frac{\text{DCG@}k}{\text{IDCG@}k}, \qquad
\text{DCG@}k = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i+1)}
\end{align}
$$

where $\text{rel}_i \in \{0, 1, 2\}$ is the relevance grade of result $i$.
NDCG@10 penalizes relevant results that appear too low in the ranking.


In [ ]:
import numpy as np

def ndcg_at_k(relevances: list[int], k: int = 10) -> float:
    """Compute NDCG@k given a list of relevance grades."""
    relevances = np.array(relevances[:k], dtype=float)
    dcg = np.sum(relevances / np.log2(np.arange(2, len(relevances) + 2)))
    ideal = np.sort(relevances)[::-1]
    idcg = np.sum(ideal / np.log2(np.arange(2, len(ideal) + 2)))
    return float(dcg / idcg) if idcg > 0 else 0.0

# Example
print(ndcg_at_k([3, 2, 3, 0, 1, 2], k=5))


## Results

JobMatchAI achieves NDCG@10 = **0.81** on JobSearch-XS, compared to **0.74** for the
BM25 baseline — a 7-point improvement while maintaining sub-100 ms median query latency.

| System | NDCG@10 | Latency (p50) |
|--------|:---:|:---:|
| BM25 only | 0.74 | 45 ms |
| BM25 + ANN (equal weights) | 0.77 | 62 ms |
| BM25 + ANN + KG (fixed weights) | 0.79 | 88 ms |
| **JobMatchAI (query-adaptive)** | **0.81** | **91 ms** |

Key finding: the **query-adaptive weight scheme** accounts for most of the gain over
fixed-weight fusion. Short queries benefit disproportionately from knowledge-graph
expansion; long free-text queries benefit more from dense semantic retrieval.


# Designing AI-Powered Business Dashboards {#sec-design}

## Lessons from JobMatchAI

The architectural decisions in JobMatchAI generalize to any AI business dashboard:

**Lesson 1 — Separate scoring from generation.** Deterministic, auditable scoring first.
LLM narration on top. Never let the LLM change the score.

**Lesson 2 — Hybrid retrieval dominates any single signal.** Sparse + dense + graph
consistently outperforms each alone. The fusion strategy (RRF, learned weights) matters
more than the individual components.

**Lesson 3 — Design for latency from the start.** Push heavy neural computation (embedding,
entity extraction) to ingestion time. Keep online inference lightweight. Sub-100 ms
is a hard business requirement for interactive dashboards.

**Lesson 4 — Explainability must be structural, not post-hoc.** Attention visualization
or LIME post-hoc explanations are insufficient for high-stakes decisions (hiring, lending,
medical triage). The explanation must derive directly from the scoring mechanism.

**Lesson 5 — Build domain-specific benchmarks.** Generic IR benchmarks (MS MARCO) do not
test the skills you care about. Disjoint skill splits and domain-specific relevance
judgments are essential for honest evaluation.


## A General Template for AI Dashboard Architecture


In [ ]:
class AIBusinessPipeline:
    """
    Template for a production AI business dashboard.
    Replace components with domain-specific implementations.
    """
    def ingest(self, raw_docs):
        """Embed, extract entities, dual-index (lexical + vector + graph)."""
        ...

    def retrieve(self, query, k=200):
        """Hybrid: BM25 + ANN + graph traversal. Fuse with RRF."""
        ...

    def rank(self, candidates, query_profile):
        """White-box utility function: named factors, adjustable weights."""
        ...

    def explain(self, ranked_results, factor_scores, kg_evidence):
        """LLM narrates pre-computed scores. Cannot modify rankings."""
        ...

    def serve(self, user_query):
        """End-to-end: enrich → retrieve → filter → rank → explain."""
        enriched = self.enrich_query(user_query)
        candidates = self.retrieve(enriched)
        filtered = self.hard_filter(candidates, enriched)
        ranked, factors = self.rank(filtered, enriched)
        explanations = [self.explain(r, factors[r], self.kg_paths(r))
                        for r in ranked[:20]]
        return ranked, explanations


## What's Next: AI Pipeline Design and Evaluation

In Lecture 2 (M07_LN2) we move from *dashboards* to *pipeline evaluation*:

- How do you measure the quality of an end-to-end AI pipeline beyond accuracy?
- How do you evaluate LLM outputs in production (hallucination, faithfulness, relevance)?
- How do you compare pipeline variants A/B — prompt, retrieval strategy, model size?
- What metrics matter for business stakeholders vs. ML engineers?


### References {.unnumbered}

::: {#refs}
:::
